# 原模型诊断
先测重复调用噪声，再测无关上下文与大 K；不预设原模型不稳定。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


先核对 OpenJev 完整产物：固定 revision 的 manifest、各文件大小和 SHA-256。只有缺失/损坏文件才重下，坏缓存入口保留为 .invalid-*；不删除 00 数据或检索缓存。Colab 从 Drive 缓存复制到本机临时目录，以普通文件加载；约需额外 9 GB 本地磁盘，复制有进度，重新连接需要重新复制，通常不需要重新下载。此格不分配 4B GPU 模型。

In [4]:
import importlib
from sa_jev import artifact_cache, model, experiments
for module in (artifact_cache, model, experiments):
    importlib.reload(module)
verified_artifact = artifact_cache.prepare_openjev_artifact(config['model']['artifact'], config['model'].get('revision'))
print('Verified OpenJev artifact:', verified_artifact)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] Invalid source: backbone/model-00001-of-00003.safetensors; actual bytes=79; expected=3991295368. Re-downloading this file at the same commit once.
[OpenJev] Preserved invalid snapshot entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2/backbone/model-00001-of-00003.safetensors.invalid-b5ab4424


backbone/model-00001-of-00003.safetensor(…): reconstructing file:   0%|          |  0.00B / 3.99GB            

backbone/model-00001-of-00003.safetensor(…): downloading bytes:           |  0.00B            

[OpenJev] model-00001-of-00003.safetensors: 1.16/3.72 GiB copied; ETA=11s
[OpenJev] model-00001-of-00003.safetensors: 2.23/3.72 GiB copied; ETA=7s
[OpenJev] model-00001-of-00003.safetensors: 3.27/3.72 GiB copied; ETA=2s
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] Invalid source: backbone/model-00002-of-00003.safetensors; actual bytes=79; expected=3979828128. Re-downloading this file at the same commit once.
[OpenJev] Preserved invalid snapshot entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2/backbone/model-00002-of-00003.safetensors.invalid-add4c50a


... (8 outputs left out)

[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-256...
[OpenJev] 9/11 tokenizer/chat_template.jinja: expected 7,756 bytes; checking SHA-256...
[OpenJev] 10/11 tokenizer/tokenizer.json: expected 19,989,325 bytes; checking SHA-256...
[OpenJev] Invalid source: tokenizer/tokenizer.json; actual bytes=79; expected=19989325. Re-downloading this file at the same commit once.
[OpenJev] Preserved invalid snapshot entry: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2/tokenizer/tokenizer.json.invalid-0a5f1fa6


tokenizer/tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer/tokenizer.json: downloading bytes:           |  0.00B            

[OpenJev] 11/11 tokenizer/tokenizer_config.json: expected 1,123 bytes; checking SHA-256...
[OpenJev] Artifact verified; repaired files=5; loading path=/tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
Verified OpenJev artifact: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411


In [ ]:
from sa_jev.experiments import diagnostics_experiment, efficiency_experiment
diagnostics_experiment(ROOT, config, limit=8 if MODE=='pilot' else 100)
efficiency_experiment(ROOT, config, limit=8 if MODE=='pilot' else 100)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [ ]:
if os.environ.get('TYPESAFE_API_KEY') or os.environ.get('JEV_API_KEY'):
    diagnostics_experiment(ROOT, config, method='dynamic_jev', limit=8 if MODE=='pilot' else 100)
    efficiency_experiment(ROOT, config, method='dynamic_jev', limit=8 if MODE=='pilot' else 100)
else:
    print('Official Jev API unavailable; no official-Jev result is reported.')

无关状态干扰使用独立且经过核验的测试包，保持不调参。未核验草稿不会被自动当成测试结果。

In [ ]:
from sa_jev.experiments import robustness_experiment
if (ROOT/'data/perturbations_reviewed.jsonl').exists():
    robustness_experiment(ROOT, config, tag='original_perturbations', methods=['openjev_frozen'] + (['dynamic_jev'] if os.environ.get('TYPESAFE_API_KEY') else []))
else:
    print('Awaiting reviewed independent perturbation packets; repeat/K diagnostics above are separate measurements.')